# Benchmark Datasets & Preprocessing PipelineThis notebook inspects the synthetic topologies and real-world evaluation datasets utilized throughout the experimental suite.

In [ ]:
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.datasets.synthetic import (
    generate_gaussian_blobs,
    generate_anisotropic_blobs,
    generate_moons,
    generate_circles,
    generate_noisy_clusters,
)
from src.datasets.loader import load_dataset_by_name
from src.datasets.preprocessing import preprocess_features, reduce_to_2d_for_visualization

In [ ]:
# 1. Synthetic Clustering Topologies
synthetic_datasets = {
    "Gaussian Blobs": generate_gaussian_blobs(n_samples=600, n_clusters=4, random_state=42)[0:2],
    "Anisotropic": generate_anisotropic_blobs(n_samples=600, n_clusters=3, random_state=42)[0:2],
    "Moons": generate_moons(n_samples=600, noise=0.08, random_state=42)[0:2],
    "Concentric Circles": generate_circles(n_samples=600, noise=0.05, random_state=42)[0:2],
    "Noisy Blobs": generate_noisy_clusters(n_samples=600, n_clusters=4, noise_ratio=0.1, random_state=42)[0:2],
}

fig, axes = plt.subplots(1, 5, figsize=(18, 3.5), dpi=150)
for ax, (name, (X, y)) in zip(axes, synthetic_datasets.items()):
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="tab10", s=12, alpha=0.6, edgecolors="none")
    ax.set_title(name, fontsize=9.5, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.4)

plt.suptitle("Synthetic Dataset Topologies", fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Canonical Real-World Benchmarks
benchmark_names = ["iris", "wine", "breast_cancer", "digits"]
summary_records = []

for name in benchmark_names:
    X_raw, y_true, meta = load_dataset_by_name(name)
    X_std, _ = preprocess_features(X_raw, method="standard")
    summary_records.append({
        "Dataset": meta["name"],
        "Samples (n)": X_raw.shape[0],
        "Features (d)": X_raw.shape[1],
        "Classes (k)": meta["n_clusters"],
        "Mean (raw)": float(np.mean(X_raw)),
        "Std (raw)": float(np.std(X_raw)),
        "Mean (standardized)": float(np.mean(X_std)),
        "Std (standardized)": float(np.std(X_std)),
    })

pd.DataFrame(summary_records)

In [ ]:
# 3. PCA Projections of Multidimensional Benchmarks
fig, axes = plt.subplots(1, 3, figsize=(14, 4.0), dpi=150)

for ax, dname in zip(axes, ["wine", "breast_cancer", "digits"]):
    X_raw, y_true, _ = load_dataset_by_name(dname)
    X_std, _ = preprocess_features(X_raw)
    X_2d, _ = reduce_to_2d_for_visualization(X_std)
    scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y_true, cmap="Set1", s=18, alpha=0.6, edgecolors="none")
    ax.set_title(f"{dname.capitalize()} (2D PCA)", fontsize=10, fontweight="bold")
    ax.set_xlabel("PC 1", fontsize=8.5)
    ax.set_ylabel("PC 2", fontsize=8.5)
    ax.grid(True, linestyle="--", alpha=0.4)

plt.suptitle("Principal Component Projections", fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()